# JailbreakBench Survey — Phase 2 / 3 runner (Colab Pro)

This notebook drives the Phase 2 (defended ASR) and Phase 3 (benign refusal
rate) experiments on a Colab Pro GPU. The target models are loaded
**locally** with `vLLM` because Together AI no longer serves Vicuna-13B-v1.5
or Llama-2-7B-chat-hf via its serverless API.

**Recommended GPU**: T4 or L4. Avoid A100 unless you actually need it — it
burns through Pro compute units 6× faster.

**Compute budget**: ~5-10 GPU-hours total for all experiments.

Before running, set a Colab secret named `TOGETHER_API_KEY` (sidebar → key
icon → Add new secret). The judge calls go to Together AI; the target-model
queries do not.

## 1. Verify GPU availability

In [ ]:
!nvidia-smi

## 2. Clone the project repository

The repo is currently private. The clone cell below uses an HTTPS URL and
will prompt for a GitHub username + Personal Access Token on first run, or
read them from `~/.netrc` if present. Once we make the repo public for the
final submission this prompt goes away.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/imzjes/jbb-survey.git'
REPO_DIR = '/content/jbb-survey'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!ls

## 3. Install dependencies (cloud + vLLM extras)

In [ ]:
!pip install --quiet -r requirements.txt -r requirements-vllm.txt
!python -c 'import jailbreakbench; from jailbreakbench.llm.vllm import LLMvLLM; print("vLLM backend OK")'

## 4. Pull the Together API key from a Colab secret

In [ ]:
import os
from google.colab import userdata
os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
assert os.environ['TOGETHER_API_KEY'], 'Set the TOGETHER_API_KEY Colab secret'
print('TOGETHER_API_KEY is set, length =', len(os.environ['TOGETHER_API_KEY']))

## 5. Phase 1 — rejudge artifacts (no GPU needed)

Cheap and fast: ~200 judge calls. Run this first as an end-to-end sanity
check before paying for the GPU phases.

In [ ]:
!python phase1_baseline_asr.py --limit 5     # smoke test

In [ ]:
!python phase1_baseline_asr.py     # full run

## 6. Phase 2 — defended ASR (GPU)

Smoke test first (5 prompts × 1 model × 1 attack × 1 defense). If this
completes cleanly, run the full sweep below.

In [ ]:
!python phase2_defense_asr.py --backend vllm \
    --models llama-2-7b-chat-hf --attacks PAIR \
    --defenses SmoothLLM --limit 5

In [ ]:
# Full Phase 2 sweep. All defenses run locally via vLLM on this Colab.
!python phase2_defense_asr.py --backend vllm \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck

## 7. Phase 3 — benign refusal rate (GPU)

In [ ]:
!python phase3_benign_refusal.py --backend vllm \
    --models llama-2-7b-chat-hf --defenses SmoothLLM --limit 5

In [ ]:
!python phase3_benign_refusal.py --backend vllm \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck \
    --include-undefended

## 8. Generate figures + LaTeX tables for the report

In [ ]:
!python make_figures.py
!ls results/

## 9. Pull results back to your laptop

Easiest: `git add results/` (the .gitignore excludes them by default — drop
the relevant files into a separate `report-assets/` dir before committing
if you want them in the repo), then `git push`. Or right-click the
`results/` folder in the Colab file browser and download as zip.